In [1]:
# ==============================================================================
# BADMINTON SHOT ANALYSIS AND CLASSIFICATION (FINAL & COMPLETE VERSION)
# ==============================================================================
# Features:
# 1. 3D Landmark Extraction with Region of Interest (ROI) to select the correct player.
# 2. Kinetic Similarity Index (KSI) with Dynamic Time Warping (DTW) for motion comparison.
# 3. Rule-based system for corrective feedback.
# 4. Deep learning (LSTM) model for shot classification.
# 5. Simplified file pathing using a single DATA_PATH.
# ==============================================================================

import cv2
import mediapipe as mp
import numpy as np
import os
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tqdm import tqdm

# ==============================================================================
# PART 1: MEDIAPIPE LANDMARK EXTRACTION (WITH ROI)
# ==============================================================================

def extract_3d_landmarks_from_video(video_path, roi_y_threshold=0.5):
    """
    Processes a video file to extract 3D pose landmarks, only considering detections
    within a specified Region of Interest (ROI) to isolate one player.
    
    Args:
        video_path (str): The full path to the input video file.
        roi_y_threshold (float): Value from 0.0 to 1.0. Only people whose center
                                 is above this vertical line will be considered.
                                 0.5 means the top half of the screen.
    
    Returns:
        np.array: An array of landmarks, or None if no valid pose is detected in the ROI.
    """
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=False, model_complexity=2, min_detection_confidence=0.5)
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error: Could not open video {video_path}")
        return None
        
    all_landmarks = []
    
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
            
        image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        
        if results.pose_landmarks:
            y_coords = [lm.y for lm in results.pose_landmarks.landmark]
            centroid_y = np.mean(y_coords)
            
            if centroid_y < roi_y_threshold: # Check if the person is in the top half (ROI)
                if results.pose_world_landmarks:
                    landmarks = results.pose_world_landmarks.landmark
                    frame_landmarks = np.array([[lm.x, lm.y, lm.z] for lm in landmarks])
                    all_landmarks.append(frame_landmarks)
            
    cap.release()
    pose.close()
    
    if not all_landmarks:
        return None
        
    return np.array(all_landmarks)

# ==============================================================================
# PART 2: KINETIC SIMILARITY INDEX (KSI) & DYNAMIC TIME WARPING (DTW)
# ==============================================================================

def dynamic_time_warping(seq1, seq2):
    """Computes the optimal alignment between two sequences using DTW."""
    n, m = len(seq1), len(seq2)
    dtw_matrix = np.full((n + 1, m + 1), np.inf)
    dtw_matrix[0, 0] = 0

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = np.linalg.norm(seq1[i - 1] - seq2[j - 1])
            last_min = min(dtw_matrix[i-1, j], dtw_matrix[i, j-1], dtw_matrix[i-1, j-1])
            dtw_matrix[i, j] = cost + last_min

    path = []
    i, j = n, m
    while i > 0 and j > 0:
        path.append((i-1, j-1))
        i, j = min((i-1, j), (i, j-1), (i-1, j-1), key=lambda x: dtw_matrix[x[0], x[1]])
    path.reverse()
    
    seq1_aligned = np.array([seq1[i] for i, j in path])
    seq2_aligned = np.array([seq2[j] for i, j in path])
    return seq1_aligned, seq2_aligned

def calculate_ksi(expert_seq, user_seq, weights={'pose': 0.4, 'velocity': 0.4, 'acceleration': 0.2}, alpha=0.1, beta=0.1):
    """Calculates the Kinetic Similarity Index (KSI) between two motion sequences."""
    expert_aligned, user_aligned = dynamic_time_warping(expert_seq, user_seq)
    
    def cosine_similarity(v1, v2):
        epsilon = 1e-8
        return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + epsilon)

    v_upper_arm_exp = expert_aligned[:, 1] - expert_aligned[:, 0]
    v_forearm_exp = expert_aligned[:, 2] - expert_aligned[:, 1]
    v_upper_arm_user = user_aligned[:, 1] - user_aligned[:, 0]
    v_forearm_user = user_aligned[:, 2] - user_aligned[:, 1]
    
    pose_sims = [(cosine_similarity(v_upper_arm_exp[i], v_upper_arm_user[i]) + 
                  cosine_similarity(v_forearm_exp[i], v_forearm_user[i])) / 2 
                 for i in range(len(v_upper_arm_exp))]
    s_pose = np.mean(pose_sims)

    wrist_exp, wrist_user = expert_aligned[:, 2], user_aligned[:, 2]
    vel_exp = np.diff(wrist_exp, axis=0, prepend=wrist_exp[0:1])
    vel_user = np.diff(wrist_user, axis=0, prepend=wrist_user[0:1])
    
    vel_sims = [cosine_similarity(vel_exp[i], vel_user[i]) * np.exp(-alpha * (np.linalg.norm(vel_exp[i]) - np.linalg.norm(vel_user[i]))**2) 
                for i in range(len(vel_exp))]
    s_velocity = np.mean(vel_sims)
    
    accel_exp = np.diff(vel_exp, axis=0, prepend=vel_exp[0:1])
    accel_user = np.diff(vel_user, axis=0, prepend=vel_user[0:1])
    
    accel_sims = [np.exp(-beta * (np.linalg.norm(accel_exp[i]) - np.linalg.norm(accel_user[i]))**2) 
                  for i in range(len(accel_exp))]
    s_acceleration = np.mean(accel_sims)

    ksi_score = (weights['pose'] * s_pose +
                 weights['velocity'] * s_velocity +
                 weights['acceleration'] * s_acceleration)

    return {'ksi_total': ksi_score, 'pose_similarity': s_pose, 'velocity_coherence': s_velocity, 'acceleration_profile': s_acceleration}

# ==============================================================================
# PART 3: DEEP LEARNING MODEL FOR CLASSIFICATION
# ==============================================================================

def load_and_preprocess_data(data_path, sequence_length=50):
    """Loads pre-extracted landmark data (.npy files) and prepares it for model training."""
    labels, sequences = [], []
    shot_types = sorted([d for d in os.listdir(data_path) if os.path.isdir(os.path.join(data_path, d))])
    label_map = {label: num for num, label in enumerate(shot_types)}
    
    for shot_type, shot_type_index in label_map.items():
        shot_path = os.path.join(data_path, shot_type)
        for file_name in os.listdir(shot_path):
            if not file_name.endswith(".npy"): continue
            
            landmarks = np.load(os.path.join(shot_path, file_name))
            hip_center = (landmarks[:, 23] + landmarks[:, 24]) / 2
            normalized_landmarks = landmarks - hip_center[:, np.newaxis, :]
            
            if len(normalized_landmarks) > sequence_length:
                normalized_landmarks = normalized_landmarks[:sequence_length]
            else:
                padding = np.zeros((sequence_length - len(normalized_landmarks), 33, 3))
                normalized_landmarks = np.concatenate([normalized_landmarks, padding])
            
            sequences.append(normalized_landmarks)
            labels.append(shot_type_index)
            
    X = np.array(sequences)
    y = to_categorical(np.array(labels)).astype(int)
    return train_test_split(X, y, test_size=0.2, random_state=42), label_map

def build_lstm_model(input_shape, num_classes):
    """Builds a simple LSTM model for sequence classification."""
    model = Sequential([
        LSTM(64, return_sequences=True, activation='relu', input_shape=input_shape),
        Dropout(0.2),
        LSTM(128, return_sequences=False, activation='relu'),
        Dropout(0.2),
        Dense(64, activation='relu'),
        Dense(num_classes, activation='softmax')])
    model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

# ==============================================================================
# PART 4: MAIN EXECUTION BLOCK
# ==============================================================================

if __name__ == "__main__":
    
    # --- MODE SELECTION ---
    MODE = "preprocess"  # Change to 'preprocess', 'train', or 'analyze'

    # --- CONFIGURATION ---
    DATA_PATH = "/home/smayan/Desktop/IPD/Data" 
    SEQUENCE_LENGTH = 50
    
    if MODE == "preprocess":
        print("--- MODE: Preprocessing videos directly in the DATA_PATH ---")
        for shot_type in os.listdir(DATA_PATH):
            shot_folder = os.path.join(DATA_PATH, shot_type)
            if not os.path.isdir(shot_folder): continue
            
            print(f"Processing shot type: {shot_type}")
            for video_file in tqdm(os.listdir(shot_folder)):
                if not video_file.endswith(".mp4"): continue

                video_file_path = os.path.join(shot_folder, video_file)
                output_file_path = os.path.join(shot_folder, f"{os.path.splitext(video_file)[0]}.npy")
                
                if os.path.exists(output_file_path): continue

                landmarks = extract_3d_landmarks_from_video(video_file_path, roi_y_threshold=0.5)
                if landmarks is not None:
                    np.save(output_file_path, landmarks)

    elif MODE == "train":
        print("--- MODE: Training shot classification model ---")
        (X_train, X_test, y_train, y_test), label_map = load_and_preprocess_data(DATA_PATH, sequence_length=SEQUENCE_LENGTH)
        
        NUM_CLASSES = len(label_map)
        INPUT_SHAPE = (SEQUENCE_LENGTH, 33 * 3)
        
        X_train = X_train.reshape(X_train.shape[0], SEQUENCE_LENGTH, -1)
        X_test = X_test.reshape(X_test.shape[0], SEQUENCE_LENGTH, -1)
        
        model = build_lstm_model(INPUT_SHAPE, NUM_CLASSES)
        print(model.summary())
        
        print("\nStarting model training...")
        model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))
        
        print("\nSaving model...")
        model.save("badminton_shot_classifier.h5")
        print("Model saved as badminton_shot_classifier.h5")

    elif MODE == "analyze":
        print("--- MODE: Analyzing a user's shot against an expert ---")
        EXPERT_VIDEO_PATH = "expert_forehand_drive.mp4"
        USER_VIDEO_PATH = "user_forehand_drive.mp4"
        
        print(f"Loading expert landmarks from {EXPERT_VIDEO_PATH}...")
        expert_landmarks = extract_3d_landmarks_from_video(EXPERT_VIDEO_PATH, roi_y_threshold=0.5)
        
        print(f"Loading user landmarks from {USER_VIDEO_PATH}...")
        user_landmarks = extract_3d_landmarks_from_video(USER_VIDEO_PATH, roi_y_threshold=0.5)

        if expert_landmarks is None or user_landmarks is None:
            print("Could not detect poses in one or both videos. Analysis aborted.")
        else:
            RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST = 12, 14, 16
            expert_elbow_seq = expert_landmarks[:, [RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST], :]
            user_elbow_seq = user_landmarks[:, [RIGHT_SHOULDER, RIGHT_ELBOW, RIGHT_WRIST], :]

            print("\nCalculating Kinetic Similarity Index (KSI)...")
            ksi_results = calculate_ksi(expert_elbow_seq, user_elbow_seq)
            
            print("\n--- Analysis Results ---")
            print(f"Overall KSI Score: {ksi_results['ksi_total']:.2f}")
            print(f"  - Postural Similarity: {ksi_results['pose_similarity']:.2f}")
            print(f"  - Velocity Coherence: {ksi_results['velocity_coherence']:.2f}")
            print(f"  - Acceleration Profile: {ksi_results['acceleration_profile']:.2f}")
            
            print("\n--- Corrective Feedback ---")
            if ksi_results['pose_similarity'] < 0.80:
                print("- Elbow Movement is inaccurate: Your arm's shape and angle are different from the expert's.")
            if ksi_results['velocity_coherence'] < 0.70:
                print("- Swing Speed: Your swing speed or path needs improvement.")
            if ksi_results['acceleration_profile'] < 0.65:
                print("- Power Generation: Focus on accelerating the racket more explosively before contact.")

    else:
        print(f"Invalid MODE selected: '{MODE}'. Please choose 'preprocess', 'train', or 'analyze'.")

2025-09-24 23:58:06.923604: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-24 23:58:06.930666: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1758738486.938764  216392 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1758738486.941188  216392 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1758738486.947618  216392 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

--- MODE: Preprocessing videos directly in the DATA_PATH ---
Processing shot type: forehand_clear


  0%|          | 0/156 [00:00<?, ?it/s]WARNING: All log messages before absl::InitializeLog() is called are written to STDERR
I0000 00:00:1758738488.166102  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758738488.206955  216546 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1758738488.241565  216519 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758738488.293266  216540 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  4%|▍         | 7/156 [00:00<00:20,  7.27it/s]I0000 00:00:1758738489.114596  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758738489.151621  

Processing shot type: forehand_lift


  0%|          | 0/174 [00:00<?, ?it/s]I0000 00:00:1758738674.150211  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758738674.193561  222263 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758738674.218994  222235 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758738674.270762  222261 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  1%|          | 1/174 [00:01<04:02,  1.40s/it]I0000 00:00:1758738675.553527  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758738675.589336  222308 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758

...

  0%|          | 0/111 [00:00<?, ?it/s]I0000 00:00:1758739364.818999  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758739364.832436  245067 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758739364.860672  245040 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758739364.920695  245044 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  1%|          | 1/111 [00:03<05:38,  3.08s/it]I0000 00:00:1758739367.896051  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758739367.912735  245156 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758

Processing shot type: forehand_drive


  0%|          | 0/159 [00:00<?, ?it/s]I0000 00:00:1758739560.373474  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758739560.383083  252127 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758739560.417719  252101 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758739560.476843  252125 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  1%|          | 1/159 [00:01<05:02,  1.91s/it]I0000 00:00:1758739562.289231  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758739562.297898  252187 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758

Processing shot type: forehand_net_shot


  0%|          | 0/105 [00:00<?, ?it/s]I0000 00:00:1758739827.241156  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758739827.250463  262537 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758739827.275113  262517 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1758739827.338138  262535 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  1%|          | 1/105 [00:02<03:40,  2.12s/it]I0000 00:00:1758739829.360745  216392 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1758739829.369078  262603 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 535.247.01), renderer: NVIDIA GeForce RTX 4070 SUPER/PCIe/SSE2
W0000 00:00:1758